# 🤖 SVM Gunshot Classifier — Training & Evaluation

This notebook loads the feature CSV generated by the trimming pipeline and:
1. Creates a **balanced 150/150 test set** (150 gunshots + 150 non-gunshots)
2. Trains an **SVM (RBF kernel)** on the remaining data
3. Reports comprehensive evaluation metrics:
   - Accuracy, Precision, Recall, F1 Score
   - Log Loss, R² Score, ROC-AUC
   - Matthews Correlation Coefficient
   - Confusion Matrix & Classification Report

In [ ]:
# Install dependencies
!pip install -q scikit-learn pandas numpy seaborn matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    log_loss,
    r2_score,
    roc_auc_score,
    roc_curve,
    matthews_corrcoef,
    ConfusionMatrixDisplay,
)

SEED = 42
np.random.seed(SEED)
print("✅ All imports loaded.")

## Step 1: Load the Dataset CSV

In [ ]:
CSV_PATH = Path(r'c:\Desktop\Data-Cleaner\Data\Output\dataset_features.csv')
df = pd.read_csv(CSV_PATH)

print(f"📊 Dataset loaded: {len(df)} samples")
print(f"   Class 0 (non-gunshot): {(df['label'] == 0).sum()}")
print(f"   Class 1 (gunshot)    : {(df['label'] == 1).sum()}")
print(f"   Features             : {len([c for c in df.columns if c.startswith('mfcc_')])}")
df.head()

## Step 2: Balanced 150/150 Train-Test Split

We sample **exactly 150 Class 0** and **exactly 150 Class 1** for the test set (300 total).  
Everything else goes into the training set.

In [ ]:
TEST_PER_CLASS = 150

df_0 = df[df['label'] == 0].copy()
df_1 = df[df['label'] == 1].copy()

# Randomly sample test sets
test_0 = df_0.sample(n=TEST_PER_CLASS, random_state=SEED)
test_1 = df_1.sample(n=TEST_PER_CLASS, random_state=SEED)
test_df = pd.concat([test_0, test_1]).sample(frac=1, random_state=SEED)

# The rest is training
train_df = df.drop(test_df.index).sample(frac=1, random_state=SEED)

# Separate features and labels
feature_cols = [c for c in df.columns if c.startswith('mfcc_')]

X_train = train_df[feature_cols].values
y_train = train_df['label'].values
X_test  = test_df[feature_cols].values
y_test  = test_df['label'].values

print(f"📦 Train set: {len(train_df)} samples  "
      f"(Class 0: {(y_train==0).sum()}, Class 1: {(y_train==1).sum()})")
print(f"📦 Test  set: {len(test_df)} samples  "
      f"(Class 0: {(y_test==0).sum()}, Class 1: {(y_test==1).sum()})")

## Step 3: Feature Scaling

SVM is sensitive to feature scales, so we standardize (zero mean, unit variance).

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)
print("✅ Features scaled with StandardScaler.")

## Step 4: 🚀 Train SVM Model

Using `SVC` with RBF kernel and `probability=True` so we can compute log loss and ROC-AUC.

In [ ]:
svm = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    probability=True,
    random_state=SEED
)

print("⏳ Training SVM ...")
svm.fit(X_train, y_train)
print("✅ SVM training complete!")
print(f"   Kernel          : {svm.kernel}")
print(f"   C               : {svm.C}")
print(f"   Support Vectors : {svm.n_support_}")

## Step 5: 📈 Evaluation Metrics

Full analysis of the model's performance on the balanced 150/150 test set.

In [ ]:
# Predictions
y_pred  = svm.predict(X_test)
y_proba = svm.predict_proba(X_test)

# Compute all metrics
acc      = accuracy_score(y_test, y_pred)
prec     = precision_score(y_test, y_pred)
rec      = recall_score(y_test, y_pred)
f1       = f1_score(y_test, y_pred)
logloss  = log_loss(y_test, y_proba)
r2       = r2_score(y_test, y_pred)
roc_auc  = roc_auc_score(y_test, y_proba[:, 1])
mcc      = matthews_corrcoef(y_test, y_pred)

# Display as a styled table
metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score',
               'Log Loss', 'R² Score', 'ROC-AUC', 'Matthews Corr. Coeff.'],
    'Score': [acc, prec, rec, f1, logloss, r2, roc_auc, mcc]
})
metrics_df['Score'] = metrics_df['Score'].round(4)

print("\n" + "="*50)
print("       📊 SVM EVALUATION METRICS")
print("="*50)
for _, row in metrics_df.iterrows():
    icon = '✅' if row['Metric'] not in ['Log Loss'] or row['Score'] < 0.5 else '⚠️'
    print(f"  {icon} {row['Metric']:<25} {row['Score']:>10.4f}")
print("="*50)
metrics_df

In [ ]:
# Full Classification Report
print("\n📋 Classification Report:\n")
print(classification_report(
    y_test, y_pred,
    target_names=['Non-Gunshot (0)', 'Gunshot (1)']
))

## Step 6: 📉 Confusion Matrix (Heatmap)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Gunshot (0)', 'Gunshot (1)'],
            yticklabels=['Non-Gunshot (0)', 'Gunshot (1)'],
            linewidths=1, linecolor='white', ax=ax)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('SVM Confusion Matrix (150/150 Test)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 7: 📈 ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba[:, 1])

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='#2563eb', lw=2.5,
        label=f'SVM (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--',
        label='Random Chance')
ax.fill_between(fpr, tpr, alpha=0.1, color='#2563eb')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — SVM Gunshot Classifier', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 8: 📊 Metrics Bar Chart

In [ ]:
# Visualize key metrics
plot_metrics = metrics_df[metrics_df['Metric'].isin(
    ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
)].copy()

colors = ['#2563eb', '#7c3aed', '#db2777', '#ea580c', '#059669']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(plot_metrics['Metric'], plot_metrics['Score'],
              color=colors, edgecolor='white', linewidth=1.5, width=0.6)

# Add value labels on bars
for bar, val in zip(bars, plot_metrics['Score']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylim(0, 1.12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('SVM Classification Metrics', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

---
## ✅ Summary

The SVM model has been trained and evaluated on a balanced 150/150 test set.  
Key outputs:
- **Metrics table** with Accuracy, Precision, Recall, F1, Log Loss, R², ROC-AUC, MCC
- **Confusion Matrix** heatmap
- **ROC Curve** with AUC
- **Metrics bar chart** for visual comparison